# 05 Model Comparison

This notebook compares Logistic Regression, XGBoost, and CatBoost using the full feature set and the fixed temporal holdout test set.

The training and holdout files are loaded from the temporal split step. Hyperparameter tuning is performed with stratified 5-fold cross-validation within the training set, using F1-score as the selection metric. The tuned models are then evaluated on the fixed temporal holdout test set using F1-score, precision, recall, accuracy, and ROC-AUC.

## 1. Experiment overview

This section summarizes the model comparison setup used in the thesis.

- **Input data:** temporal training and holdout files generated by the split notebook
- **Feature set:** full feature set after preprocessing and sparsity-based exclusions
- **Models:** Logistic Regression, XGBoost, and CatBoost
- **Validation strategy:** stratified 5-fold cross-validation within the training set
- **Hyperparameter search:** grid search with F1-score as the selection metric
- **Final evaluation:** fixed temporal holdout test set
- **Threshold:** 0.5 for threshold-dependent metrics
- **Reported metrics:** F1-score, precision, recall, accuracy, and ROC-AUC

The holdout test set is not used for preprocessing fitting, hyperparameter tuning, or model selection.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
import catboost

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)


## 2. Project paths and experiment settings

This section defines the input files, output locations, and shared modeling settings.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
LOGS_DIR = OUTPUTS_DIR / "logs"
MODELS_DIR = OUTPUTS_DIR / "models"

MODEL_COMPARISON_TABLES_DIR = TABLES_DIR / "model_comparison"
MODEL_COMPARISON_LOG_DIR = LOGS_DIR / "model_comparison"
MODEL_COMPARISON_MODEL_DIR = MODELS_DIR / "model_comparison"

X_TRAIN_PATH = INTERIM_DIR / "X_train_temporal.csv"
X_TEST_PATH = INTERIM_DIR / "X_test_temporal.csv"
Y_TRAIN_PATH = INTERIM_DIR / "y_train_temporal.csv"
Y_TEST_PATH = INTERIM_DIR / "y_test_temporal.csv"

TARGET_COL = "target"
RANDOM_SEED = 2026
CV_FOLDS = 5
THRESHOLD = 0.5
SCORING = "f1"

CATEGORICAL_COLUMNS = ["country_code", "market"]

GRIDSEARCH_N_JOBS = -1
GRIDSEARCH_VERBOSE = 1

OUTPUT_PATHS = {
    "results": MODEL_COMPARISON_TABLES_DIR / "model_comparison_results.csv",
    "predictions": MODEL_COMPARISON_TABLES_DIR / "model_comparison_holdout_predictions.csv",
    "best_params": MODEL_COMPARISON_TABLES_DIR / "model_comparison_best_params.csv",
    "selected_features": MODEL_COMPARISON_TABLES_DIR / "model_comparison_selected_features.csv",
    "cv_summary": MODEL_COMPARISON_TABLES_DIR / "model_comparison_cv_summary.csv",
    "run_metadata": MODEL_COMPARISON_TABLES_DIR / "model_comparison_run_metadata.json",
}

for directory in [MODEL_COMPARISON_TABLES_DIR, MODEL_COMPARISON_LOG_DIR, MODEL_COMPARISON_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## 3. Hyperparameter search spaces

This section defines the grid-search spaces reported in the methodology section.

In [3]:
LR_PARAMS = {
    "model__penalty": ["l2"],
    "model__C": [0.01, 0.1, 1, 10, 100],
}

XGB_PARAMS = {
    "model__n_estimators": [100, 300, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 1.0],
    "model__colsample_bytree": [0.7, 1.0],
}

CATBOOST_PARAMS = {
    "model__iterations": [200, 500, 800],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__depth": [4, 6, 8],
    "model__l2_leaf_reg": [1, 3, 5, 7],
}

search_space_summary = pd.DataFrame([
    {"model": "logistic_regression", "n_parameter_settings": np.prod([len(v) for v in LR_PARAMS.values()])},
    {"model": "xgboost", "n_parameter_settings": np.prod([len(v) for v in XGB_PARAMS.values()])},
    {"model": "catboost", "n_parameter_settings": np.prod([len(v) for v in CATBOOST_PARAMS.values()])},
])
search_space_summary["n_cv_fits"] = search_space_summary["n_parameter_settings"] * CV_FOLDS
display(search_space_summary)

,model,n_parameter_settings,n_cv_fits
0,logistic_regression,5,25
1,xgboost,324,1620
2,catboost,108,540


## 4. Helper functions

This section defines functions for loading data, constructing pipelines, tuning models, evaluating holdout performance, and saving grid-search logs.

In [4]:
def read_target(path: Path, target_col: str = TARGET_COL) -> pd.Series:
    """Read a target CSV file and return the target column as an integer Series."""
    target_df = pd.read_csv(path)
    if target_col not in target_df.columns:
        raise ValueError(f"Target column '{target_col}' was not found in {path}. Available columns: {list(target_df.columns)}")
    return target_df[target_col].astype(int)


def check_required_files(paths: list[Path]) -> None:
    """Raise a clear error if an expected input file is missing."""
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "The following required input files are missing:\n"
            + "\n".join(missing)
            + "\n\nRun preprocessing and temporal split first, or update the paths in Section 2."
        )


def load_temporal_split_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Load the fixed temporal train/test split produced by 04_temporal_split.ipynb."""
    required_paths = [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH]
    check_required_files(required_paths)

    X_train = pd.read_csv(X_TRAIN_PATH)
    X_test = pd.read_csv(X_TEST_PATH)
    y_train = read_target(Y_TRAIN_PATH)
    y_test = read_target(Y_TEST_PATH)

    return X_train, X_test, y_train, y_test


def select_columns(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    selected_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    """Select features that exist in both train and test, and report missing features."""
    present = [column for column in selected_columns if column in X_train.columns and column in X_test.columns]
    missing = [column for column in selected_columns if column not in X_train.columns or column not in X_test.columns]

    if not present:
        raise ValueError("None of the selected features were found in both X_train and X_test.")

    return X_train[present].copy(), X_test[present].copy(), present, missing


def make_one_hot_encoder() -> OneHotEncoder:
    """Create a OneHotEncoder that works across recent scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def make_preprocessor(
    X_train: pd.DataFrame,
    categorical_columns: list[str],
    scale_numeric: bool = False,
) -> ColumnTransformer:
    """Build the preprocessing pipeline fitted inside each CV fold."""
    categorical_present = [column for column in categorical_columns if column in X_train.columns]
    numeric_present = [column for column in X_train.columns if column not in categorical_present]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="other")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_present),
            ("categorical", categorical_pipeline, categorical_present),
        ],
        remainder="drop",
    )


def make_model_pipeline(
    model_name: str,
    X_train: pd.DataFrame,
    categorical_columns: list[str],
    random_seed: int = RANDOM_SEED,
) -> Pipeline:
    """Create a model-specific sklearn pipeline."""
    if model_name == "logistic_regression":
        return Pipeline([
            ("preprocessor", make_preprocessor(X_train, categorical_columns, scale_numeric=True)),
            ("model", LogisticRegression(max_iter=5000, random_state=random_seed)),
        ])

    if model_name == "xgboost":
        return Pipeline([
            ("preprocessor", make_preprocessor(X_train, categorical_columns, scale_numeric=False)),
            ("model", XGBClassifier(
                random_state=random_seed,
                eval_metric="logloss",
                n_jobs=1,
                tree_method="hist",
            )),
        ])

    if model_name == "catboost":
        return Pipeline([
            ("preprocessor", make_preprocessor(X_train, categorical_columns, scale_numeric=False)),
            ("model", CatBoostClassifier(
                random_seed=random_seed,
                verbose=0,
                allow_writing_files=False,
            )),
        ])

    raise ValueError(f"Unknown model name: {model_name}")


def make_stratified_cv_folds(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    n_splits: int = CV_FOLDS,
    random_seed: int = RANDOM_SEED,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Create explicit stratified CV folds for inspection and GridSearchCV."""
    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_seed,
    )
    return list(splitter.split(X_train, y_train))


def run_grid_search(
    model_name: str,
    estimator: Pipeline,
    param_grid: dict,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cv_folds: list[tuple[np.ndarray, np.ndarray]],
) -> GridSearchCV:
    """Run GridSearchCV and return the fitted search object."""
    print(f"\nStarting grid search for {model_name}...")
    print(f"Number of parameter settings: {np.prod([len(v) for v in param_grid.values()])}")
    print(f"Number of CV folds: {len(cv_folds)}")

    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=SCORING,
        cv=cv_folds,
        n_jobs=GRIDSEARCH_N_JOBS,
        verbose=GRIDSEARCH_VERBOSE,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    print(f"Finished grid search for {model_name}.")
    print(f"Best CV F1: {search.best_score_:.4f}")
    print(f"Best parameters: {search.best_params_}")

    return search


def evaluate_binary_classifier(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    threshold: float = THRESHOLD,
) -> tuple[dict, pd.DataFrame]:
    """Evaluate a fitted binary classifier on the fixed holdout test set."""
    y_prob = fitted_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "threshold": threshold,
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

    predictions = pd.DataFrame({
        "y_true": y_test.to_numpy(),
        "y_pred": y_pred,
        "y_prob": y_prob,
    })

    return metrics, predictions


def summarize_best_params(model_name: str, search: GridSearchCV) -> list[dict]:
    """Create a readable table of compared and selected hyperparameter values."""
    rows = []
    for parameter, compared_values in search.param_grid.items():
        rows.append({
            "model": model_name,
            "parameter": parameter.replace("model__", ""),
            "compared_values": ", ".join(map(str, compared_values)),
            "best_value": search.best_params_.get(parameter),
            "best_cv_f1": search.best_score_,
        })
    return rows


def summarize_cv_results(model_name: str, search: GridSearchCV) -> dict:
    """Summarize the best CV row, including mean and standard deviation."""
    cv_df = pd.DataFrame(search.cv_results_)
    best_row = cv_df.loc[search.best_index_]

    return {
        "model": model_name,
        "best_cv_f1_mean": search.best_score_,
        "best_cv_f1_std": best_row.get("std_test_score", np.nan),
        "best_cv_rank": int(best_row.get("rank_test_score", 1)),
    }


def save_grid_search_log(model_name: str, search: GridSearchCV) -> Path:
    """Save the full GridSearchCV results for auditability."""
    output_path = MODEL_COMPARISON_LOG_DIR / f"{model_name}_gridsearch_log.csv"
    pd.DataFrame(search.cv_results_).to_csv(output_path, index=False)
    return output_path


## 5. Load the fixed train-test split

This section loads the temporal training and holdout files produced by the split notebook.

In [5]:
X_train_raw, X_test_raw, y_train, y_test = load_temporal_split_inputs()

split_shape_summary = pd.DataFrame([
    {"split": "train", "feature_rows": X_train_raw.shape[0], "feature_columns": X_train_raw.shape[1], "target_rows": y_train.shape[0]},
    {"split": "holdout", "feature_rows": X_test_raw.shape[0], "feature_columns": X_test_raw.shape[1], "target_rows": y_test.shape[0]},
])

display(split_shape_summary)
display(X_train_raw.head())


,split,feature_rows,feature_columns,target_rows
0,train,4563,22,4563
1,holdout,1141,22,1141


,_original_index,funding_total_usd,seed,venture,angel,debt_financing,private_equity,undisclosed,founded_year,funding_rounds,...,is_single_round,has_round_a,has_round_b,has_round_c,has_round_d,has_round_e,country_code,market,is_usa,last_funding_at
0,886,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1991.00,1.00,...,1,0,0,0,0,0,USA,other,1,1993-08-30
1,3310,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1994.00,1.00,...,1,0,0,0,0,0,USA,News,1,1995-04-01
2,4224,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1996.00,1.00,...,1,0,0,0,0,0,other,other,0,1996-10-01
3,5446,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1996.00,1.00,...,1,0,0,0,0,0,USA,Biotechnology,1,1996-10-01
4,1913,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1990.00,1.00,...,1,0,0,0,0,0,USA,other,1,1996-12-05


## 6. Check target distribution

This section summarizes the acquired/closed class distribution after the temporal split.

In [6]:
def target_distribution(y: pd.Series, label: str) -> pd.DataFrame:
    counts = y.value_counts(dropna=False).sort_index()
    proportions = y.value_counts(normalize=True, dropna=False).sort_index()
    return pd.DataFrame({
        "split": label,
        "outcome": counts.index,
        "count": counts.values,
        "proportion": proportions.values,
    })

target_dist = pd.concat([
    target_distribution(y_train, "train"),
    target_distribution(y_test, "holdout"),
], ignore_index=True)

display(target_dist.round(2))


,split,outcome,count,proportion
0,train,0,1957,0.43
1,train,1,2606,0.57
2,holdout,0,360,0.32
3,holdout,1,781,0.68


## 7. Select the full feature set

This section selects the final predictors used for the model comparison.

In [7]:
feature_blocks = {
    "funding_scale": [
        "funding_total_usd",
        "seed",
        "venture",
        "angel",
        "debt_financing",
        "private_equity",
        "undisclosed",
    ],
    "funding_dynamics": [
        "founded_year",
        "funding_rounds",
        "funding_duration_days",
        "avg_days_between_rounds",
        "is_single_round",
    ],
    "stage_structure": [
        "has_round_a",
        "has_round_b",
        "has_round_c",
        "has_round_d",
        "has_round_e",
    ],
    "geography": ["country_code"],
    "industry": ["market"],
}

full_feature_set = [feature for features in feature_blocks.values() for feature in features]

excluded_features_check = [
    "grant",
    "convertible_note",
    "product_crowdfunding",
    "equity_crowdfunding",
    "round_f",
    "round_g",
    "round_h",
    "post_ipo_equity",
    "post_ipo_debt",
    "secondary_market",
    "is_usa",
    "first_funding_at",
    "last_funding_at",
]

unexpected_selected_features = [
    feature for feature in excluded_features_check
    if feature in full_feature_set
]

if unexpected_selected_features:
    raise ValueError(
        "The selected feature set contains variables that should be excluded: "
        f"{unexpected_selected_features}."
    )

X_train, X_test, selected_features, missing_features = select_columns(
    X_train=X_train_raw,
    X_test=X_test_raw,
    selected_columns=full_feature_set,
)

selected_feature_table = pd.DataFrame({
    "feature": selected_features,
    "feature_group": [
        next(group for group, features in feature_blocks.items() if feature in features)
        for feature in selected_features
    ],
})

feature_block_summary = (
    selected_feature_table
    .groupby("feature_group", as_index=False)
    .agg(n_features=("feature", "count"))
)

display(feature_block_summary)
display(selected_feature_table)

if missing_features:
    raise ValueError(f"Expected features missing from the split files: {missing_features}")


,feature_group,n_features
0,funding_dynamics,5
1,funding_scale,7
2,geography,1
3,industry,1
4,stage_structure,5


,feature,feature_group
0,funding_total_usd,funding_scale
1,seed,funding_scale
2,venture,funding_scale
3,angel,funding_scale
4,debt_financing,funding_scale
5,private_equity,funding_scale
6,undisclosed,funding_scale
7,founded_year,funding_dynamics
8,funding_rounds,funding_dynamics
9,funding_duration_days,funding_dynamics


## 8. Check feature composition and missing values

This section confirms the categorical and numeric inputs before fitting the model pipelines.

In [8]:
categorical_present = [column for column in CATEGORICAL_COLUMNS if column in X_train.columns]
numeric_present = [column for column in X_train.columns if column not in categorical_present]

feature_type_table = pd.DataFrame({
    "feature": selected_features,
    "type": ["categorical" if feature in categorical_present else "numeric" for feature in selected_features],
})

missing_summary = pd.DataFrame({
    "feature": selected_features,
    "train_missing_ratio": X_train[selected_features].isna().mean().values,
    "holdout_missing_ratio": X_test[selected_features].isna().mean().values,
}).sort_values(["train_missing_ratio", "holdout_missing_ratio"], ascending=False)

display(feature_type_table)
display(missing_summary.round(2))


,feature,type
0,funding_total_usd,numeric
1,seed,numeric
2,venture,numeric
3,angel,numeric
4,debt_financing,numeric
5,private_equity,numeric
6,undisclosed,numeric
7,founded_year,numeric
8,funding_rounds,numeric
9,funding_duration_days,numeric


,feature,train_missing_ratio,holdout_missing_ratio
0,funding_total_usd,0.00,0.00
1,seed,0.00,0.00
2,venture,0.00,0.00
3,angel,0.00,0.00
4,debt_financing,0.00,0.00
5,private_equity,0.00,0.00
6,undisclosed,0.00,0.00
7,founded_year,0.00,0.00
8,funding_rounds,0.00,0.00
9,funding_duration_days,0.00,0.00


## 9. Create stratified cross-validation folds

This section creates the stratified 5-fold cross-validation splits used for hyperparameter tuning.

In [9]:
cv_folds = make_stratified_cv_folds(X_train, y_train)

fold_rows = []
for fold_id, (train_idx, valid_idx) in enumerate(cv_folds, start=1):
    y_valid = y_train.iloc[valid_idx]
    fold_rows.append({
        "fold": fold_id,
        "cv_train_size": len(train_idx),
        "cv_valid_size": len(valid_idx),
        "valid_positive_count": int((y_valid == 1).sum()),
        "valid_negative_count": int((y_valid == 0).sum()),
        "valid_positive_rate": float((y_valid == 1).mean()),
    })

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary)

,fold,cv_train_size,cv_valid_size,valid_positive_count,valid_negative_count,valid_positive_rate
0,1,3650,913,522,391,0.57
1,2,3650,913,521,392,0.57
2,3,3650,913,521,392,0.57
3,4,3651,912,521,391,0.57
4,5,3651,912,521,391,0.57


## 10. Build model pipelines

This section defines the model-specific sklearn pipelines. Logistic Regression uses numeric scaling, while XGBoost and CatBoost use the same imputation and one-hot encoding without numeric scaling.

In [10]:
model_specs = {
    "Logistic Regression": {
        "pipeline": make_model_pipeline("logistic_regression", X_train, CATEGORICAL_COLUMNS),
        "params": LR_PARAMS,
    },
    "XGBoost": {
        "pipeline": make_model_pipeline("xgboost", X_train, CATEGORICAL_COLUMNS),
        "params": XGB_PARAMS,
    },
    "CatBoost": {
        "pipeline": make_model_pipeline("catboost", X_train, CATEGORICAL_COLUMNS),
        "params": CATBOOST_PARAMS,
    },
}

pipeline_summary = pd.DataFrame([
    {"model": model_name, "n_parameter_settings": int(np.prod([len(v) for v in spec["params"].values()]))}
    for model_name, spec in model_specs.items()
])

display(pipeline_summary)


,model,n_parameter_settings
0,Logistic Regression,5
1,XGBoost,324
2,CatBoost,108


## 11. Run hyperparameter tuning

This section runs grid search for each model using F1-score as the selection metric.

In [11]:
searches = {}
cv_log_paths = {}

for model_name, spec in model_specs.items():
    searches[model_name] = run_grid_search(
        model_name=model_name,
        estimator=spec["pipeline"],
        param_grid=spec["params"],
        X_train=X_train,
        y_train=y_train,
        cv_folds=cv_folds,
    )
    cv_log_paths[model_name] = save_grid_search_log(model_name, searches[model_name])



Starting grid search for Logistic Regression...
Number of parameter settings: 5
Number of CV folds: 5
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Finished grid search for Logistic Regression.
Best CV F1: 0.7421
Best parameters: {'model__C': 0.1, 'model__penalty': 'l2'}

Starting grid search for XGBoost...
Number of parameter settings: 324
Number of CV folds: 5
Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Finished grid search for XGBoost.
Best CV F1: 0.7673
Best parameters: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.01, 'model__max_depth': 7, 'model__min_child_weight': 1, 'model__n_estimators': 100, 'model__subsample': 0.7}

Starting grid search for CatBoost...
Number of parameter settings: 108
Number of CV folds: 5
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Finished grid search for CatBoost.
Best CV F1: 0.7610
Best parameters: {'model__depth': 8, 'model__iterations': 200, 'model__l2_leaf_reg': 7, 'model__learning_r

## 12. Evaluate tuned models on the fixed holdout test set

This section evaluates each tuned model on the fixed temporal holdout test set.

In [12]:
results = []
prediction_frames = []
best_param_rows = []
cv_summary_rows = []

for model_name, search in searches.items():
    metrics, predictions = evaluate_binary_classifier(
        fitted_model=search.best_estimator_,
        X_test=X_test,
        y_test=y_test,
        threshold=THRESHOLD,
    )

    metrics["model"] = model_name
    results.append(metrics)

    predictions["model"] = model_name
    prediction_frames.append(predictions)

    best_param_rows.extend(summarize_best_params(model_name, search))
    cv_summary_rows.append(summarize_cv_results(model_name, search))

results_df = (
    pd.DataFrame(results)
    .loc[:, [
        "model",
        "f1",
        "precision",
        "recall",
        "accuracy",
        "roc_auc",
        "true_negative",
        "false_positive",
        "false_negative",
        "true_positive",
    ]]
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)

best_params_df = (
    pd.DataFrame(best_param_rows)
    .drop(columns=["best_cv_f1"], errors="ignore")
)

cv_summary_df = pd.DataFrame(cv_summary_rows)

thesis_results_table = results_df[[
    "model",
    "f1",
    "precision",
    "recall",
    "accuracy",
    "roc_auc",
]].copy()

display(thesis_results_table.round(2))
display(best_params_df)


,model,f1,precision,recall,accuracy,roc_auc
0,XGBoost,0.81,0.79,0.82,0.73,0.77
1,CatBoost,0.77,0.82,0.73,0.71,0.77
2,Logistic Regression,0.67,0.84,0.56,0.63,0.72


,model,parameter,compared_values,best_value
0,Logistic Regression,penalty,l2,l2
1,Logistic Regression,C,"0.01, 0.1, 1, 10, 100",0.10
2,XGBoost,n_estimators,"100, 300, 500",100
3,XGBoost,learning_rate,"0.01, 0.05, 0.1",0.01
4,XGBoost,max_depth,"3, 5, 7",7
5,XGBoost,min_child_weight,"1, 3, 5",1
6,XGBoost,subsample,"0.7, 1.0",0.70
7,XGBoost,colsample_bytree,"0.7, 1.0",0.70
8,CatBoost,iterations,"200, 500, 800",200
9,CatBoost,learning_rate,"0.01, 0.05, 0.1",0.05


## 13. Save outputs

This section saves the model comparison results, holdout predictions, selected features, cross-validation summary, selected hyperparameters, grid-search logs, fitted estimators, and run metadata.

In [13]:
OUTPUT_PATHS["results"].parent.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_COMPARISON_MODEL_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUTPUT_PATHS["results"], index=False)
predictions_df.to_csv(OUTPUT_PATHS["predictions"], index=False)
best_params_df.to_csv(OUTPUT_PATHS["best_params"], index=False)
selected_feature_table.to_csv(OUTPUT_PATHS["selected_features"], index=False)
cv_summary_df.to_csv(OUTPUT_PATHS["cv_summary"], index=False)

saved_model_paths = {}
for model_name, search in searches.items():
    model_path = MODEL_COMPARISON_MODEL_DIR / f"{model_name.lower().replace(' ', '_')}_best_estimator.joblib"
    joblib.dump(search.best_estimator_, model_path)
    saved_model_paths[model_name] = str(model_path)

run_metadata = {
    "target_col": TARGET_COL,
    "random_seed": RANDOM_SEED,
    "cv_folds": CV_FOLDS,
    "scoring": SCORING,
    "threshold": THRESHOLD,
    "categorical_columns": categorical_present,
    "selected_features": selected_features,
    "missing_features": missing_features,
    "input_paths": {
        "X_train": str(X_TRAIN_PATH),
        "X_test": str(X_TEST_PATH),
        "y_train": str(Y_TRAIN_PATH),
        "y_test": str(Y_TEST_PATH),
    },
    "output_paths": {key: str(value) for key, value in OUTPUT_PATHS.items()},
    "cv_log_paths": {key: str(value) for key, value in cv_log_paths.items()},
    "saved_model_paths": saved_model_paths,
}
with open(OUTPUT_PATHS["run_metadata"], "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
